In [6]:
import os
import numpy as np
import pandas as pd

import allel

from mpire import WorkerPool

VCF_PATH = "/share/home/zhoujb/project/riceDataComb/processData/1-0-6-barley_600/processData/geneHap/"
HAPDF_PATH = "/share/home/zhoujb/project/riceDataComb/processData/1-0-6-barley_600/processData/2-allGeneHapDf/"
HAPPER_PATH = "/share/home/zhoujb/project/riceDataComb/processData/1-0-6-barley_600/processData/2-allGeneHapPer/"

In [7]:
def allelVCF2HapPer(gene_id):
    # 读取VCF文件
    vcf_file = os.path.join(VCF_PATH, "{}.vcf".format(gene_id))
    
    if not os.path.exists(vcf_file):
        return
        
    callset = allel.read_vcf(vcf_file)

    samples = callset['samples']
    #samples = [x.split("_")[0] for x in samples]

    gt = allel.GenotypeArray(callset['calldata/GT'])
    chrom = callset['variants/CHROM']
    pos = callset['variants/POS']

    # 将 GT（二倍体）转换为 “1维基因型编码”
    # 0→ref/ref, 1→杂合, 2→alt/alt
    geno = gt.to_n_alt()   # shape = (num_snps, num_samples)
    geno = geno.T                # 转换成样本 × SNP 矩阵

    geno_df = pd.DataFrame(geno, columns=pos, index=samples)
    geno_df.index.name = "sample"

    # 将每个样本的 SNP 型组合成 tuple，这样可作为 dict key
    hap_strings = [tuple(row) for row in geno]
    
    hap_df = pd.DataFrame({'sample': samples,'haplotype': hap_strings})

    # 确定 unique haplotypes，并编号
    unique_haps = hap_df['haplotype'].unique()
    hap_to_id = {h: f"Hap{idx+1}" for idx, h in enumerate(unique_haps)}

    hap_df['haplotype_id'] = hap_df['haplotype'].map(hap_to_id)

    freq = hap_df.groupby(['haplotype_id']).size().reset_index(name='count')
    freq['percentage'] = freq['count'] / freq['count'].sum()
    freq_d = freq.set_index('haplotype_id').to_dict()
    freq2count = freq_d["count"]
    freq2per = freq_d["percentage"]

    hap_df["count"] = hap_df['haplotype_id'].map(freq2count)
    hap_df["percentage"] = hap_df['haplotype_id'].map(freq2per)

    sample2hap = hap_df[["sample", "haplotype_id"]].set_index("sample").to_dict()["haplotype_id"]
    geno_df["haplotype_id"] = geno_df.index.map(sample2hap)

    geno_df.to_csv(os.path.join(HAPDF_PATH, "{}.txt".format(gene_id)), sep="\t")
    hap_df[['sample', 'haplotype_id', 'count', 'percentage']].to_csv(os.path.join(HAPPER_PATH, "{}.txt".format(gene_id)), sep="\t", index=0)

    print("{} DONE".format(gene_id))

    return


if __name__ == "__main__":
    in_file_path = "/share/home/zhoujb/project/riceDataComb/processData/1-0-6-barley_600/rawData/RiceHom/best_hits_with_coords.tsv"
    info_file = pd.read_table(in_file_path)
    all_gene_list = info_file["gene_symbol"].to_list()

    with WorkerPool(n_jobs=48) as pool:
        pool.map(allelVCF2HapPer, all_gene_list, progress_bar=True)

    print("ALL DONE")

  0%|                                                                                                                               | 0/141 [00:00<?, ?it/s]

HORVU.MOREX.r3.3HG0222080 DONE
HORVU.MOREX.r3.6HG0603410 DONEHORVU.MOREX.r3.6HG0561180 DONE

HORVU.MOREX.r3.4HG0386330 DONE
HORVU.MOREX.r3.2HG0125030 DONEHORVU.MOREX.r3.3HG0309700 DONEHORVU.MOREX.r3.2HG0189750 DONE


HORVU.MOREX.r3.7HG0749940 DONEHORVU.MOREX.r3.5HG0527890 DONE

HORVU.MOREX.r3.7HG0657900 DONE


  7%|████████▎                                                                                                             | 10/141 [00:00<00:01, 99.84it/s]

HORVU.MOREX.r3.1HG0022500 DONE
HORVU.MOREX.r3.5HG0446560 DONEHORVU.MOREX.r3.1HG0023370 DONE

HORVU.MOREX.r3.7HG0676380 DONEHORVU.MOREX.r3.6HG0560710 DONE

HORVU.MOREX.r3.2HG0166450 DONE
HORVU.MOREX.r3.4HG0344990 DONEHORVU.MOREX.r3.1HG0053170 DONE

HORVU.MOREX.r3.4HG0389730 DONE
HORVU.MOREX.r3.3HG0230520 DONE
HORVU.MOREX.r3.2HG0185220 DONE
HORVU.MOREX.r3.3HG0230060 DONEHORVU.MOREX.r3.7HG0695720 DONE

HORVU.MOREX.r3.4HG0346720 DONEHORVU.MOREX.r3.4HG0400410 DONE
HORVU.MOREX.r3.1HG0074250 DONE



 18%|█████████████████████▌                                                                                               | 26/141 [00:00<00:00, 135.07it/s]

...

HORVU.MOREX.r3.7HG0658800 DONE
HORVU.MOREX.r3.4HG0348500 DONE
HORVU.MOREX.r3.2HG0101230 DONE
HORVU.MOREX.r3.4HG0404020 DONE
HORVU.MOREX.r3.1HG0094260 DONE
HORVU.MOREX.r3.5HG0519420 DONE
HORVU.MOREX.r3.2HG0203510 DONE
HORVU.MOREX.r3.1HG0068750 DONE
HORVU.MOREX.r3.7HG0639770 DONE
HORVU.MOREX.r3.6HG0540900 DONE
HORVU.MOREX.r3.1HG0081630 DONE
HORVU.MOREX.r3.3HG0222050 DONE
HORVU.MOREX.r3.4HG0413540 DONE
HORVU.MOREX.r3.7HG0654930 DONE
HORVU.MOREX.r3.5HG0469040 DONE
HORVU.MOREX.r3.5HG0498170 DONEHORVU.MOREX.r3.1HG0027890 DONE

HORVU.MOREX.r3.7HG0744530 DONE


 94%|████████████████████████████████████████████████████████████████████████████████████████████████████████████▌       | 132/141 [00:01<00:00, 123.27it/s]

HORVU.MOREX.r3.5HG0502370 DONE
HORVU.MOREX.r3.2HG0173670 DONE
HORVU.MOREX.r3.3HG0251630 DONE
HORVU.MOREX.r3.1HG0000210 DONE
HORVU.MOREX.r3.1HG0064740 DONE
HORVU.MOREX.r3.1HG0003210 DONE
HORVU.MOREX.r3.3HG0220340 DONE
HORVU.MOREX.r3.5HG0526470 DONE


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 141/141 [00:04<00:00, 123.27it/s]


ALL DONE
